In [1]:
!pip install requests

In [2]:
import requests
import pprint
from getpass import getpass


In [3]:
# APIキーを指定
api_key = getpass('APIキーを入力してください: ')

APIキーを入力してください: ··········


In [4]:
# 検索キーワードを変数search_stationに格納
search_station = input('指定の駅から半径500メートル以内のレストランを検索します。: ')

指定の駅から半径500メートル以内のレストランを検索します。: 名古屋駅


In [15]:
import urllib.parse

# 日本語search_stationをURLエンコードする
encoded_address = urllib.parse.quote(search_station)

# リクエスト用URLの作成
url = f"https://maps.googleapis.com/maps/api/geocode/json?address={encoded_address}&key={api_key}"

# APIへアクセスしてJSONを取得
response = requests.get(url)
station = response.json()

# 緯度、経度を取得
if station["status"] == "OK":
  location = station["results"][0]["geometry"]["location"]
  lat = location["lat"]  # 緯度
  lng = location["lng"]  # 経度
  print(f"緯度: {lat}, 経度: {lng}")
else:
  print(f"エラー: {station['status']}")

緯度: 35.170915, 経度: 136.8815369


In [16]:
# PlacesAPI(NEW)を使って情報取得
url = "https://places.googleapis.com/v1/places:searchNearby"

headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": api_key,
    "X-Goog-FieldMask": "places.displayName,places.formattedAddress,places.rating"
}

data = {
    "includedTypes": ["restaurant"],
    "maxResultCount": 10,
    "languageCode": "ja",
    "locationRestriction": {
        "circle": {
            "center": {
                "latitude": lat,
                "longitude": lng
            },
            "radius": 500.0
        }
    }
}

response = requests.post(url, headers=headers, json=data)
json_data = response.json()

"""
# PlacesAPIを使って情報取得 <------ ＜＜旧が使えないので新で行う＞＞
url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"

# データを取得
response = requests.get(
    url,
    params={
        'key': api_key,  # APIキーをパラメータに追加
        'latitude': lat,  # 緯度
        'longitude': lng,  # 経度
        'language': 'ja',  # 日本語
        'radius': 500,  # 半径
        'type': 'restaurant',  # 半径
        'maxResults': 10  # 取得する動画の最大数を指定
    }
)
"""


'\n# PlacesAPIを使って情報取得 <------ ＜＜旧が使えないので新で行う＞＞\nurl = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"\n\n# データを取得\nresponse = requests.get(\n    url,\n    params={\n        \'key\': api_key,  # APIキーをパラメータに追加\n        \'latitude\': lat,  # 緯度\n        \'longitude\': lng,  # 経度\n        \'language\': \'ja\',  # 日本語\n        \'radius\': 500,  # 半径\n        \'type\': \'restaurant\',  # 半径\n        \'maxResults\': 10  # 取得する動画の最大数を指定\n    }\n)\n'

In [17]:
# jsonの中身確認
pprint.pprint(json_data)

{'places': [{'displayName': {'languageCode': 'ja', 'text': 'ミッドランドスクエア'},
             'formattedAddress': '日本、〒450-0002 愛知県名古屋市中村区名駅４丁目７−１',
             'rating': 3.9},
            {'displayName': {'languageCode': 'ja', 'text': '一蘭 名古屋駅店'},
             'formattedAddress': '日本、〒450-0002 愛知県名古屋市中村区名駅３丁目２５−６ ホテルリソル名古屋 '
                                 'B1F',
             'rating': 4.1},
            {'displayName': {'languageCode': 'ja', 'text': '獅子丸'},
             'formattedAddress': '日本、〒453-0013 愛知県名古屋市中村区亀島２丁目１−１',
             'rating': 4.3},
            {'displayName': {'languageCode': 'ja', 'text': '名鉄グランドホテル'},
             'formattedAddress': '日本、〒450-0002 愛知県名古屋市中村区名駅１丁目２−４ 名鉄バスターミナルビル '
                                 '１１階～１８階',
             'rating': 3.9},
            {'displayName': {'languageCode': 'ja', 'text': '味仙 JR名古屋駅店'},
             'formattedAddress': '日本、〒450-0003 愛知県名古屋市中村区名駅１丁目１−４ JR名古屋駅 '
                                 'うまいもん通り',
             'rating': 3.8

In [18]:
# 出力
for place in json_data.get("places", []):
    name = place.get("displayName", {}).get("text", "店名なし")
    address = place.get("formattedAddress", "住所なし")
    rating = place.get("rating", "評価なし")
    print(f"店名: {name}")
    print(f"住所: {address}")
    print(f"評価: {rating}")
    print("---")



店名: ミッドランドスクエア
住所: 日本、〒450-0002 愛知県名古屋市中村区名駅４丁目７−１
評価: 3.9
---
店名: 一蘭 名古屋駅店
住所: 日本、〒450-0002 愛知県名古屋市中村区名駅３丁目２５−６ ホテルリソル名古屋 B1F
評価: 4.1
---
店名: 獅子丸
住所: 日本、〒453-0013 愛知県名古屋市中村区亀島２丁目１−１
評価: 4.3
---
店名: 名鉄グランドホテル
住所: 日本、〒450-0002 愛知県名古屋市中村区名駅１丁目２−４ 名鉄バスターミナルビル １１階～１８階
評価: 3.9
---
店名: 味仙 JR名古屋駅店
住所: 日本、〒450-0003 愛知県名古屋市中村区名駅１丁目１−４ JR名古屋駅 うまいもん通り
評価: 3.8
---
店名: 炭焼うな富士 名駅店
住所: 日本、〒450-0002 愛知県名古屋市中村区名駅４丁目８−１８ 名古屋三井ビルディング北館 B1F
評価: 4.4
---
店名: 馬喰一代 名古屋WEST
住所: 日本、〒451-0046 愛知県名古屋市西区牛島町６−２４ アクロスキューブ名古屋 2F
評価: 4.2
---
店名: マクドナルド 名古屋ゲートウォーク店
住所: 日本、〒450-0002 愛知県名古屋市中村区名駅１丁目１−２ ゲートウォーク Ｂ１Ｆ
評価: 3.4
---
店名: まるや本店 JR名古屋駅店
住所: 日本、〒450-0002 愛知県名古屋市中村区名駅１丁目１−４ 名古屋うまいもん通り
評価: 4.3
---
店名: 矢場とん 名古屋駅エスカ店
住所: 日本、〒453-0015 愛知県名古屋市中村区椿町６−９ エスカ地下街
評価: 4
---
